## User Dashboard: Household Energy Recommendations

The dashboard brings together the full pipeline into a single interactive interface. It is built with Streamlit and combines the XGBoost price forecast, model explainability outputs, weather context, and an AI-generated reasoning summary. The goal is to translate model outputs into something a household can act on — showing when prices are expected to be high or low, why, and which hours are best for running flexible loads.

The dashboard is launched from the terminal:

```
streamlit run dashboard.py
```

It reads pre-computed files from `outputs/model/` and `data/` and does not re-run any models at runtime.

### 5-day price forecast

The main panel shows the XGBoost 5-day ahead price forecast for DK1. Five day cards display the average predicted price per day in DKK/kWh, each with an arrow indicating whether that day is expected to be more or less expensive than today. Below the cards, a full hourly chart shows the 120-hour forecast together with a ±1σ confidence band derived from historical walk-forward residuals. A history window of actual prices is shown to the left to give context for how the forecast compares to recent observed levels.

The sidebar date picker lets the user select any date that was covered by the walk-forward cross-validation, so the forecast for any past issue time can be replayed and compared against what actually happened.

In [ ]:
import pandas as pd
from src.analysis.model_uncertainty import plot_single_forecast

preds_df = pd.read_parquet("outputs/model/predictions.parquet")

# Replicates the dashboard's hourly forecast chart for the latest available issue time
plot_single_forecast(preds_df)

The black line shows the actual realised prices and the blue dashed line shows the XGBoost prediction. The coloured background bands mark which day model is responsible for each section of the forecast — Day 1 through Day 5. Error tends to increase with horizon, which is expected given that weather uncertainty and price volatility both grow further into the future.

### Model insights: feature importance and SHAP

The dashboard includes a model insights panel that helps explain what is driving the price forecasts. Two charts are shown side by side for the Day 1 model (h = 1–24).

The feature importance chart shows the top features by XGBoost's built-in split-based importance score. This gives a quick overview of which variables the model relies on most, but it does not indicate the direction of the effect.

The SHAP beeswarm chart addresses this by showing the contribution of each feature to individual predictions. Each point is one sample — its horizontal position shows how much that feature pushed the prediction up or down, and its colour shows whether the feature value was high (red) or low (blue) for that sample. Together the two charts show both which features matter and how they affect the forecast.

In [ ]:
import joblib
from src.models.XG_Boost_full_Res import OUTPUT_DIR
from src.analysis.explainable_ai import plot_feature_importance

final_models = joblib.load(OUTPUT_DIR / "final_day_models.joblib")

plot_feature_importance(final_models)

In [ ]:
from src.analysis.explainable_ai import plot_shap_explanations

# Loads and displays the saved SHAP bar and beeswarm figures
plot_shap_explanations()

### Local explanations: LIME

While SHAP provides a global view of feature importance across many predictions, LIME explains a single forecast at a specific hour. It fits a simple linear model locally around one data point and identifies which features pushed that particular prediction up or down. In the dashboard this is shown for the midpoint hour of each forecast day, giving a snapshot of the key drivers for that specific issue time.

This is useful for the recommendation layer — if a high price is forecast on Day 3, LIME can show whether it is driven by low wind, high temperature, or a recent run of expensive prices, which helps make the AI-generated reasoning more grounded.

In [ ]:
from src.analysis.explainable_ai import plot_lime_explanations

plot_lime_explanations()

### Weather explorer

The weather explorer section shows the last 7 days of actual weather observations followed by the 5-day simulated forecast for wind speed, solar radiation, and temperature. It is aligned to the same issue time as the price forecast so the user can see the weather context that the model was working with when the price forecast was made.

This section is useful for understanding the price forecast — a predicted price drop on Day 3 is easier to interpret when the weather panel shows a sharp increase in wind speed on the same day.

In [ ]:
from src.data.data_processing import plot_weather_forecast

plot_weather_forecast(ctx_days=7)

### AI-generated reasoning

The dashboard includes two AI-generated summaries powered by Groq, both triggered by a button press.

The first is a **forecast reasoning** written in the style of a Nordic electricity market analyst. It receives the daily predicted prices, per-day weather averages, and intraday peak and cheap hours, and produces four bullet points explaining the key price drivers across the 5-day window — for example linking a mid-week price spike to a forecast drop in wind production.

The second is a **household recommendation** written as a friendly energy advisor. It receives the same price forecast together with the three cheapest identified time windows, and produces five practical bullet points covering when to run flexible loads such as EV charging, the dishwasher, washing machine, and heat pump.

Neither summary receives any SHAP or LIME output — the AI works entirely from the pre-summarised price and weather numbers passed in the prompt.